In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
#Datasets and DataLoaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

BUILD CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10),
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x

In [8]:
model = CNN()


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

TRAINNING CNN

In [14]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()              # Reset gradients
        
        outputs = model(images)            # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        
        loss.backward()                    # Backpropagation  <-- MISSING
        optimizer.step()                   # Update weights
        
        epoch_training_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs}, "
          f"Loss: {epoch_training_loss/len(trainloader):.4f}")

Epoch 1/10, Loss: 0.9722
Epoch 2/10, Loss: 0.7843
Epoch 3/10, Loss: 0.6443
Epoch 4/10, Loss: 0.5360
Epoch 5/10, Loss: 0.4355
Epoch 6/10, Loss: 0.3599
Epoch 7/10, Loss: 0.2765
Epoch 8/10, Loss: 0.2143
Epoch 9/10, Loss: 0.1692
Epoch 10/10, Loss: 0.1352


In [15]:
#evaluate our CNN

correct_labels = 0
total_labels =0
model.eval()
with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)
        correct_labels +=(predicted == labels).sum().item()
        total_labels +=labels.size(0)
    print(f"accuracy ={correct_labels/total_labels*100}")

accuracy =74.11999999999999


In [16]:
#add validation loss and plot graph for training loss and validation loss

In [17]:
torch.save(model.state_dict(), "cnn_model.pt")